# In-Class Assignment: LASSO vs. Linear Regression

**Question:** can LASSO predict a student's final math grade better than ordinary linear regression (OLS), using the original inputs only (no squares or interactions)?

Fill in every `# TODO` and every **Your answer** cell. Run all cells before you submit. Your submitted notebook should read as a short report: someone who has not seen the assignment should be able to follow it. Grades in this data run from 0 to 20.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BLUE, ORANGE = "#2a78d6", "#eb6834"   # LASSO, OLS
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9", "axes.axisbelow": True})

df = pd.read_csv("https://bryantstats.github.io/math460/data/student_performance.csv")
y = df["G3"]
X = df.drop(columns="G3")


def make_prep():
    """Standardize the numeric inputs. Turn each text input into 0/1 columns."""
    return make_column_transformer(
        (StandardScaler(), make_column_selector(dtype_include="number")),
        (OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
         make_column_selector(dtype_exclude="number")),
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


def scores(model, X, y):
    """Return R-squared and RMSE of a fitted model on the data given."""
    pred = model.predict(X)
    return r2_score(y, pred), mean_squared_error(y, pred) ** 0.5

## Part 1. Introduce the dataset

Use the data dictionary in the handout. Run the code below, then describe the data.

In [ ]:
# TODO: print the number of rows and the number of input columns
# TODO: print how many inputs are numeric and how many are text
# TODO: draw a histogram of the final grade G3 (one bar per grade)
# TODO: print the correlation of G1 and G2 with G3


**Your answer.** In your own words: what does one row represent, what are we predicting, and what kinds of inputs do we have? Is anything in the histogram unusual? Which inputs do you expect to matter most, and why?

*Write your answer here.*

## Part 2. Explain the testing procedure

Use these settings for both models:

| Setting | Value |
|---|---|
| Number of train/test splits | 50 (`random_state` = 1, 2, ..., 50) |
| Split size | 70% training, 30% testing |
| Tuning $\alpha$ | 5-fold cross-validation on the training data only (`LassoCV`) |
| Scoring | Test $R^2$ and test RMSE |

In [ ]:
# TODO: use make_prep() on the full X to find how many columns the models actually see


**Your answer.** Explain the procedure so that someone else could repeat it. Be sure to answer:

a. How many students are used for training and for testing on each split? How many columns do the models see, and why is it more than the number of inputs in the raw data?

b. Why test on 50 splits rather than 1?

c. Why must $\alpha$ be chosen without using the test data?

d. Why do both models use the same splits?

*Write your answer here.*

## Part 3. OLS results, tuning $\alpha$, and LASSO coefficients

This part uses one split, with `random_state=1`.

### 3a. OLS

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [ ]:
# TODO: fit make_pipeline(make_prep(), LinearRegression()) on the training data; call it ols
# TODO: print its training and test R-squared and RMSE
# TODO: show the 10 largest OLS coefficients by absolute size
#       names: ols[0].get_feature_names_out()   values: ols[-1].coef_


### 3b. Tuning $\alpha$ for LASSO

In [ ]:
# TODO: fit make_pipeline(make_prep(), LassoCV(cv=5, random_state=1, max_iter=50000)); call it lasso
# TODO: plot the cross-validated MSE against alpha, with a log x-axis
#       alpha values: lasso[-1].alphas_     average MSE: lasso[-1].mse_path_.mean(axis=1)
# TODO: mark the chosen alpha, lasso[-1].alpha_, with a vertical line
# TODO: print the chosen alpha, its training and test R-squared and RMSE


**Your answer.** Describe the shape of the curve. What happens to the model when $\alpha$ is too small, and when it is too large?

*Write your answer here.*

### 3c. Plot the LASSO coefficients

Make two plots: how the coefficients change with $\alpha$, and the coefficients at the chosen $\alpha$.

In [ ]:
# Prepared training data, for the path plot
Z_train = make_prep().fit_transform(X_train)
alphas = np.logspace(-3, 0.7, 40)

# TODO: fit Lasso(alpha=a, max_iter=50000) on (Z_train, y_train) for each a in alphas and save .coef_
# TODO: plot every coefficient against alpha (log x-axis), with a vertical line at the chosen alpha
# TODO: draw a bar chart of only the nonzero coefficients at the chosen alpha
#       use lasso[-1].coef_ with the column names
# TODO: print how many columns LASSO kept, out of how many


**Your answer.** What happens to the coefficients as $\alpha$ grows? Which column is the last to reach zero, and what does that tell you?

*Write your answer here.*

## Part 4. Compare OLS and tuned LASSO

Now repeat the comparison on all 50 splits. LASSO must be re-tuned on every split.

In [ ]:
rows, ols_coefs, lasso_coefs = [], {}, {}
for seed in range(1, 51):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=seed)

    # TODO: fit OLS and LassoCV on (Xa, ya). Call them m_ols and m_lasso
    # (use the same settings as Part 3)

    names_s = m_ols[0].get_feature_names_out()
    ols_coefs[seed] = pd.Series(m_ols[-1].coef_, index=names_s)
    lasso_coefs[seed] = pd.Series(m_lasso[-1].coef_, index=names_s)
    rows.append({"seed": seed,
                 "ols_train": scores(m_ols, Xa, ya)[0], "lasso_train": scores(m_lasso, Xa, ya)[0],
                 "ols_r2": scores(m_ols, Xb, yb)[0], "lasso_r2": scores(m_lasso, Xb, yb)[0],
                 "ols_rmse": scores(m_ols, Xb, yb)[1], "lasso_rmse": scores(m_lasso, Xb, yb)[1],
                 "alpha": m_lasso[-1].alpha_, "kept": (m_lasso[-1].coef_ != 0).sum()})
res = pd.DataFrame(rows).set_index("seed")
ols_coefs, lasso_coefs = pd.DataFrame(ols_coefs), pd.DataFrame(lasso_coefs)

# TODO: make a table with one row per model showing: median test R2, worst test R2,
#       median test RMSE, and the average gap (train R2 minus test R2)
# TODO: print how many of the 50 splits LASSO had the higher test R2
# TODO: plot LASSO test R2 against OLS test R2 for the 50 splits, with the line y = x


**Your answer.** Which model predicts better, by how much, and how consistently? Translate the RMSE into grade points. Is the difference large enough to matter? Why is this evidence stronger than the single split in Part 3?

*Write your answer here.*

## Part 5. Summarize the variables in the two models

In [ ]:
# TODO: make a table with one row per column, showing
#         - the OLS coefficient and the LASSO coefficient from the Part 3 split
#         - pct_kept: the percent of the 50 splits where LASSO's coefficient was not zero
#           hint: (lasso_coefs != 0).mean(axis=1) * 100
#       Sort it by pct_kept, largest first, and show it
# TODO: print how many columns OLS uses, how many LASSO keeps on the Part 3 split,
#       how many are kept on all 50 splits, and how many on none of them
# TODO: compare absences for students with G3 = 0 and with G3 > 0


**Your answer.**

a. Which variables does LASSO keep almost every time? Do they make sense?

b. Pick one variable with a noticeable OLS coefficient that LASSO sets to zero on the Part 3 split. Why might LASSO drop it? (Check how many students are in each group, if it is a text variable.)

c. Does LASSO keep a positive coefficient on `absences`? Does that mean missing school raises final grades? Use your output to suggest what else could be going on.

*Write your answer here.*

## Part 6. Explain why LASSO works better

In [ ]:
# TODO: for each column, compute the standard deviation of its coefficient over the 50 splits
#       (ols_coefs and lasso_coefs have one column per split)
#       print the median of those standard deviations for each model


**Your answer.** In one paragraph, explain why LASSO beats OLS on this dataset. Quote numbers from your own output as evidence. Your explanation must address:

- what most of the columns add once `G1` and `G2` are known;
- what estimating all of those columns costs OLS on about 276 training students (use the train-test gap and the coefficient standard deviations);
- what LASSO gives up in exchange (use training $R^2$), and why that trade pays off on test data.

*Write your answer here.*